## Imports and project paths

Import the required packages, locate the repository from the notebook directory, and define the input and output folders used throughout the workflow.

In [ ]:
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import py4dgeo

repo_dir = Path.cwd().parent if Path.cwd().name == "jupyter" else Path.cwd()

data_dir = repo_dir / "kijkduin"
pointcloud_dir = data_dir / "pointclouds"
results_dir = repo_dir / "results"
tables_dir = repo_dir / "tables"
figures_dir = repo_dir / "figures"

for path in (results_dir, tables_dir, figures_dir):
    path.mkdir(exist_ok=True)

if not pointcloud_dir.is_dir():
    raise FileNotFoundError(f"Point-cloud directory not found: {pointcloud_dir}")

print(f"Point-cloud directory: {pointcloud_dir}")

## Epoch files and timestamps

Discover the LAZ epochs, extract acquisition timestamps from the filenames, and separate the first epoch as the reference from the remaining comparison epochs.

In [ ]:
pc_list = sorted(path.name for path in pointcloud_dir.glob("*.laz"))

if not pc_list:
    raise FileNotFoundError("No LAZ files found.")

timestamps = []
for file_name in pc_list:
    timestamp_string = "_".join(Path(file_name).stem.split("_")[1:])
    try:
        timestamps.append(datetime.strptime(timestamp_string, "%y%m%d_%H%M%S"))
    except ValueError as exc:
        raise ValueError(f"Could not extract timestamp from: {file_name}") from exc

reference_epoch_file = pointcloud_dir / pc_list[0]
reference_timestamp = timestamps[0]

comparison_epoch_files = [pointcloud_dir / name for name in pc_list[1:]]
comparison_timestamps = timestamps[1:]

metadata = pd.DataFrame({
    "epoch_index": np.arange(len(pc_list)),
    "filename": pc_list,
    "timestamp": timestamps,
})

display(metadata.head())
print(f"Number of epochs: {len(pc_list)}")

## Spatiotemporal analysis

Create a new `SpatiotemporalAnalysis`, load the first epoch as the reference point cloud, and assign its acquisition timestamp.

In [ ]:
analysis_file = data_dir / "kijkduin.zip"

if analysis_file.exists():
    analysis_file.unlink()

analysis = py4dgeo.SpatiotemporalAnalysis(str(analysis_file), force=True)

reference_epoch = py4dgeo.read_from_las(str(reference_epoch_file))
reference_epoch.timestamp = reference_timestamp
analysis.reference_epoch = reference_epoch

print(f"Reference epoch: {reference_epoch_file.name}")
print(f"Reference points: {reference_epoch.cloud.shape[0]}")

## Corepoints and M3C2 configuration

Use the reference point cloud as the corepoints and configure M3C2 with the parameters used to derive the surface-change time series.

In [ ]:
normal_radii = (5.0,)
cyl_radius = 1.0
max_distance = 10.0
registration_error = 0.019

analysis.corepoints = reference_epoch.cloud
analysis.m3c2 = py4dgeo.M3C2(
    normal_radii=normal_radii,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

print(f"Corepoints: {analysis.corepoints.cloud.shape[0]}")
print(
    f"M3C2: normal_radii={normal_radii}, cyl_radius={cyl_radius}, "
    f"max_distance={max_distance}, registration_error={registration_error}"
)

## Comparison epochs

Load each remaining LAZ epoch, assign its timestamp, and add it to the spatiotemporal analysis. Adding the epochs computes the M3C2 distance and uncertainty time series.

In [ ]:
if len(comparison_epoch_files) != len(comparison_timestamps):
    raise ValueError("Comparison epochs and timestamps do not match.")

start_time = time.time()

for index, (epoch_file, timestamp) in enumerate(
    zip(comparison_epoch_files, comparison_timestamps), start=1
):
    epoch = py4dgeo.read_from_las(str(epoch_file))
    epoch.timestamp = timestamp
    analysis.add_epochs(epoch)

    print(f"{index:03d}/{len(comparison_epoch_files)} {epoch_file.name}")

elapsed = time.time() - start_time

print(f"\nDistances shape: {analysis.distances.shape}")
print(f"Uncertainties shape: {analysis.uncertainties.shape}")
print(f"Elapsed time: {elapsed:.2f} s")

## Temporal smoothing

Apply temporal averaging to the M3C2 distance time series using a 14-epoch window and reconstruct the acquisition timestamps used in the subsequent segmentation analysis.

In [ ]:
smoothing_window = 14

timestamps_analysis = [
    analysis.reference_epoch.timestamp + timedelta
    for timedelta in analysis.timedeltas
]

analysis.smoothed_distances = py4dgeo.temporal_averaging(
    analysis.distances,
    smoothing_window=smoothing_window,
)

print(f"Raw M3C2 shape: {analysis.distances.shape}")
print(f"Smoothed M3C2 shape: {analysis.smoothed_distances.shape}")
print(f"Number of timestamps: {len(timestamps_analysis)}")